# Amostra derivada e anonimizada do SAMU-MG (v2) · para compartilhar com o professor
Lê a tabela plana e **sem identificadores** gerada pelo notebook `entrega1_qualidade_samu_v2` (parâmetro `FLAT_TABLE`)
e gera um CSV pequeno **sem horários absolutos**: durações em minutos, indicadores de preenchimento, sinais de qualidade e faixas.
**Só publique depois de autorização formal (chefia, ISIS e encarregado de dados da SES-MG).**

In [ ]:
# CONFIGURAÇÃO ---------------------------------------------------------------
SOURCE = "catalogo.schema.ocorrencias_flat"   # AJUSTAR: tabela criada com FLAT_TABLE no notebook entrega1_qualidade_samu_v2
OUT_DIR = "/Workspace/Users/SEU_EMAIL/amostra_anonimizada"   # AJUSTAR: pasta do workspace ou Volume (para baixar o CSV)

MAX_POR_CONSORCIO = 5000     # linhas por consórcio (equilibra tamanhos; consórcios menores entram inteiros)
K_MIN = 5                    # tamanho mínimo de grupo (k-anonimato)
SEED = 42
SECRET_SCOPE, SECRET_KEY = "samu", "salt_amostra"   # opcional; se não existir, usa um segredo aleatório descartado ao fim

In [ ]:
import os, secrets
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import Window

os.makedirs(OUT_DIR, exist_ok=True)
try:
    SALT = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)
except Exception:
    SALT = secrets.token_hex(32)   # segredo aleatório desta execução: as chaves ficam irreversíveis
    print("Secret scope não encontrado: usando segredo aleatório desta execução (recomendado até).")

df = spark.table(SOURCE)
df = df.toDF(*[c.lower() for c in df.columns])

MARCOS_T = ["datacriacao", "datatarm", "dataregulador", "dataradiooperador", "pj9", "pj10", "sj9", "sj10"]
GPS = ["pj9", "pj10", "sj9", "sj10"]
for c in MARCOS_T:
    df = df.withColumn(c, F.col(c).cast("timestamp"))

## 1. Um atendimento por linha e derivação (nenhum horário absoluto, nenhuma coordenada, nenhum nome sai daqui)

In [ ]:
df = (df.withColumn("chave_chamada", F.concat_ws("|", "idsamu", F.year("datacriacao").cast("string"), "numocorrencia"))
        .withColumn("chave_atend", F.concat_ws("|", "idsamu", "idocorrencia")))
n_pre = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in MARCOS_T)
w = Window.partitionBy("chave_atend").orderBy(n_pre.desc(), F.col("datacriacao").asc())
at = (df.where(F.col("idocorrencia").isNotNull())
        .withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn"))

def hash_key(col):
    return F.substring(F.sha2(F.concat(F.lit(SALT), F.col(col)), 256), 1, 16)

def minutos(a, b):
    return F.round((F.unix_timestamp(b) - F.unix_timestamp(a)) / 60.0, 2)

def coord_ok(m):
    return (F.col(f"lat_{m}").between(-23.0, -14.0) & F.col(f"long_{m}").between(-52.0, -39.0))

def hav_km(a, b):
    p1, p2 = F.radians(F.col(f"lat_{a}")), F.radians(F.col(f"lat_{b}"))
    x = F.sin((p2 - p1) / 2) ** 2 + F.cos(p1) * F.cos(p2) * F.sin(F.radians(F.col(f"long_{b}") - F.col(f"long_{a}")) / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(x))

PARES = {
    "d_criacao_tarm": ("datacriacao", "datatarm"),
    "d_tarm_regulador": ("datatarm", "dataregulador"),
    "d_regulador_radio": ("dataregulador", "dataradiooperador"),
    "d_criacao_radio": ("datacriacao", "dataradiooperador"),   # T1
    "d_radio_pj9": ("dataradiooperador", "pj9"),
    "d_pj9_pj10": ("pj9", "pj10"),                             # T2 efetivo
    "d_radio_pj10": ("dataradiooperador", "pj10"),             # T2
    "d_pj10_sj9": ("pj10", "sj9"),
    "d_sj9_sj10": ("sj9", "sj10"),                             # T3 efetivo
    "d_pj10_sj10": ("pj10", "sj10"),                           # T3
}

def gps_flag(m):   # 1 = coordenada nula (lat 0); 0 = coordenada presente; nulo = marco ausente
    return (F.when(F.col(m).isNull(), None)
             .when(F.col(f"lat_{m}") == 0, 1).otherwise(0))

out = at.select(
    hash_key("chave_chamada").alias("id_chamada_pseudo"),
    hash_key("chave_atend").alias("id_atend_pseudo"),
    F.col("consorcio"),
    F.year("datacriacao").alias("ano"),
    F.dayofweek("datacriacao").alias("dia_semana"),   # 1 = domingo ... 7 = sábado
    F.when(F.hour("datacriacao").between(0, 5), "00-05").when(F.hour("datacriacao").between(6, 11), "06-11")
     .when(F.hour("datacriacao").between(12, 17), "12-17").otherwise("18-23").alias("faixa_horaria"),
    F.upper(F.col("codigo")).alias("codigo"),
    F.col("comatendimento"), F.col("tipounidade"), F.col("tipotransporte"),
    F.col("hospitaldestino").isNotNull().cast("int").alias("tem_transporte"),   # o nome do hospital não é publicado
    *[minutos(a, b).alias(n) for n, (a, b) in PARES.items()],
    *[F.col(c).isNotNull().cast("int").alias(f"tem_{c}") for c in MARCOS_T],
    *[(F.minute(c) % 10).alias(f"dig_min_{c}") for c in MARCOS_T],
    *[F.when(F.col(c).isNotNull(), (F.second(c) == 0).cast("int")).alias(f"seg0_{c}") for c in MARCOS_T],
    *[gps_flag(m).alias(f"coord_nula_{m}") for m in GPS],
    F.when(coord_ok("pj9") & coord_ok("pj10"), F.round(hav_km("pj9", "pj10"), 1)).alias("dist_km_pj9_pj10"),
    F.when(coord_ok("sj9") & coord_ok("sj10"), F.round(hav_km("sj9", "sj10"), 1)).alias("dist_km_sj9_sj10"),
    F.when(F.col("idade").isNull(), "nao_informado").when(F.col("idade") == 0, "zero_registrado")
     .when((F.col("idade") < 0) | (F.col("idade") > 120), "fora_0_120").when(F.col("idade") <= 11, "0-11")
     .when(F.col("idade") <= 17, "12-17").when(F.col("idade") <= 59, "18-59").otherwise("60+").alias("faixa_etaria"),
    F.upper(F.col("sexo")).alias("sexo"),
    F.upper(F.col("obito")).alias("obito"),
    F.col("tipoobito").alias("tipo_obito"),
)

## 2. Amostra equilibrada por consórcio

In [ ]:
w_c = Window.partitionBy("consorcio").orderBy(F.rand(SEED))
amostra = out.withColumn("_r", F.row_number().over(w_c)).where(F.col("_r") <= MAX_POR_CONSORCIO).drop("_r")
n_antes = amostra.count()
print("Linhas antes do k-anonimato:", n_antes)

## 3. k-anonimato: generaliza e, se ainda restar grupo raro, remove

In [ ]:
QUASI = ["consorcio", "ano", "faixa_etaria", "sexo", "tem_transporte", "obito", "tipo_obito"]
w_q = Window.partitionBy(*[F.col(c) for c in QUASI])
amostra = amostra.withColumn("_k", F.count("*").over(w_q))
for c in ["faixa_etaria", "sexo", "tipo_obito"]:
    amostra = amostra.withColumn(c, F.when(F.col("_k") < K_MIN, F.lit("suprimido")).otherwise(F.col(c)))
w_q2 = Window.partitionBy(*[F.col(c) for c in QUASI])
amostra = (amostra.drop("_k").withColumn("_k", F.count("*").over(w_q2)).where(F.col("_k") >= K_MIN).drop("_k"))
n_final = amostra.count()
print(f"Linhas finais: {n_final} | removidas por k-anonimato: {n_antes - n_final} ({100 * (n_antes - n_final) / max(n_antes, 1):.1f}%)")
if n_antes and (n_antes - n_final) / n_antes > 0.10:
    print("ATENÇÃO: mais de 10% removido. Considere aumentar MAX_POR_CONSORCIO ou revisar QUASI.")

## 4. Trava de segurança e exportação

In [ ]:
PROIBIDO = ["nom", "tel", "ender", "lat_", "long_", "hospital", "placa", "cpf", "cns", "logradouro", "bairro"]
suspeitas = [c for c in amostra.columns if any(p in c.lower() for p in PROIBIDO)]
assert not suspeitas, f"Colunas suspeitas na saída: {suspeitas}"

pdf = amostra.toPandas()
pdf.to_csv(f"{OUT_DIR}/amostra_samu_derivada.csv", index=False)
print("Linhas exportadas:", len(pdf), "| consórcios:", pdf["consorcio"].nunique())
display(pdf.groupby("consorcio").size().rename("linhas").reset_index())

## 5. Dicionário de dados

In [ ]:
dic = pd.DataFrame([
    ("id_chamada_pseudo", "Chave da chamada (IdSamu + ano + NumOcorrencia) embaralhada com segredo; irreversível sem o segredo"),
    ("id_atend_pseudo", "Chave do atendimento (IdSamu + IdOcorrencia) embaralhada"),
    ("consorcio", "Consórcio (nome da tabela de origem)"),
    ("ano", "Ano de abertura da ocorrência"),
    ("dia_semana", "Dia da semana da abertura (1 = domingo)"),
    ("faixa_horaria", "Faixa de 6 h da hora de abertura (provisória; não é o turno institucional)"),
    ("codigo", "Prioridade ou tipo da chamada (ex.: vermelho, amarelo, verde, orientação, trote)"),
    ("comatendimento", "S se houve atendimento com recurso, N caso contrário"),
    ("tipounidade", "USB ou USA (vazio se não houve recurso)"),
    ("tipotransporte", "APH ou transferência (vazio se não houve transporte)"),
    ("tem_transporte", "1 se há hospital de destino registrado (o nome do hospital não é publicado)"),
    ("d_criacao_tarm", "Minutos: criação até TARM"),
    ("d_tarm_regulador", "Minutos: TARM até regulador"),
    ("d_regulador_radio", "Minutos: regulador até acionamento pelo rádio operador"),
    ("d_criacao_radio", "Minutos: T1 (criação até acionamento)"),
    ("d_radio_pj9", "Minutos: mobilização (acionamento até início do deslocamento)"),
    ("d_pj9_pj10", "Minutos: T2 efetivo (início do deslocamento até chegada à cena)"),
    ("d_radio_pj10", "Minutos: T2 (acionamento até chegada à cena)"),
    ("d_pj10_sj9", "Minutos: tempo em cena (chegada à cena até saída para o hospital)"),
    ("d_sj9_sj10", "Minutos: T3 efetivo (saída da cena até chegada ao hospital)"),
    ("d_pj10_sj10", "Minutos: T3 (chegada à cena até chegada ao hospital)"),
    ("tem_<marco>", "1 se o marco de tempo está preenchido (datacriacao, datatarm, dataregulador, dataradiooperador, pj9, pj10, sj9, sj10)"),
    ("dig_min_<marco>", "Último dígito do minuto do marco (para teste de preferência por dígito)"),
    ("seg0_<marco>", "1 se o segundo do marco é zero; vazio se o marco não existe"),
    ("coord_nula_<pj9|pj10|sj9|sj10>", "1 se a coordenada gravada no marco é nula (latitude 0); 0 se presente; vazio se o marco não existe"),
    ("dist_km_pj9_pj10", "Distância em linha reta (km, 1 casa) entre os marcos PJ9 e PJ10, só com coordenadas válidas em MG"),
    ("dist_km_sj9_sj10", "Distância em linha reta (km, 1 casa) entre SJ9 e SJ10, só com coordenadas válidas em MG"),
    ("faixa_etaria", "Faixa etária, com categorias de qualidade (nao_informado, zero_registrado, fora_0_120) e 'suprimido'"),
    ("sexo", "Sexo registrado, ou 'suprimido'"),
    ("obito", "Óbito registrado (S ou N)"),
    ("tipo_obito", "Tipo de óbito registrado, ou 'suprimido'"),
], columns=["campo", "descricao"])
dic.to_csv(f"{OUT_DIR}/dicionario_dados.csv", index=False)
print("Arquivos gerados em", OUT_DIR)